# gpu-free-serve — Spark-X2.6-4B at 128k (Kaggle)

Boots [XHToken/Spark-X2.6-4B-GGUF](https://huggingface.co/XHToken/Spark-X2.6-4B-GGUF)
on this notebook's free GPU and exposes an OpenAI-compatible endpoint.

**Before running:** Settings > Accelerator > GPU T4 x2 (and Internet: On)

Why this notebook exists: the model is a custom architecture
(`Spark2_5ForCausalLM`, repo-supplied `modeling_spark.py`), which vLLM cannot
execute, and its quantised releases elsewhere are MLX (Apple-silicon only — an
`mlx`/`safetensors` repo will never load on a T4). GGUF is the format CUDA can
actually run, so this is the Ollama (llama.cpp) route: download the GGUF,
`ollama create` it with a 128k context, then serve it.

Quant choice. This notebook uses **Q8_0 (4.07 GiB)**, the closest GGUF gets to
the 6-bit/8-bit weights people reach for on other stacks. Others in the same
repo, if you want to trade size for speed:

* `Spark-X2.6-4B-Q4_K_M.gguf` — 3.30 GiB, fastest, smallest KV headroom.
* `Spark-X2.6-4B.gguf` — 7.66 GiB, essentially unquantised; still fits a T4 at
  128k with a q8_0 cache (~10 GiB total), just slower per token.

The context is cheap here: the model is native to 1M positions, only 9 of its
36 layers are full attention (27 are sliding-window at 512 tokens) and it has 4
KV heads, so a 128k cache costs ~4.6 GiB f16 / ~2.3 GiB at q8_0.

In [ ]:
!nvidia-smi

In [ ]:
# Clone the repo (or update it if it is already here).
import os, subprocess

os.chdir("/kaggle/working")
if os.path.isdir("gpu-free-serve"):
    subprocess.run(["git", "-C", "gpu-free-serve", "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/mufflegyro/gpu-free-serve.git"], check=True)
os.chdir("/kaggle/working/gpu-free-serve")
print(os.getcwd(), subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())

In [ ]:
# Secrets are read from the platform's secret manager and never written into
# this notebook. Kaggle: Add-ons > Secrets (and turn Internet ON in the right sidebar).
#
#   GITHUB_TOKEN     scope `gist` -> lets `gpufree connect --gist` work
#   HF_TOKEN         only for gated models (Llama, Gemma, ...)
#   NGROK_AUTHTOKEN  only if TUNNEL = "ngrok"
#   GPUFREE_API_KEY  optional fixed API key; otherwise a random one is generated
import os
from kaggle_secrets import UserSecretsClient

_client = UserSecretsClient()

def read_secret(name):
    try:
        return _client.get_secret(name)
    except Exception:
        return None

for name in ("GITHUB_TOKEN", "HF_TOKEN", "NGROK_AUTHTOKEN", "GPUFREE_API_KEY"):
    value = read_secret(name)
    if value:
        os.environ[name] = value
        print(f"{name}: loaded")
    else:
        print(f"{name}: not set (fine if you don't need it)")

## Import the GGUF into Ollama

Downloads ~4 GiB of weights, then creates a local model with `num_ctx` baked
into it (so the context survives restarts and applies to every request).

`ollama create` needs a running daemon, so this starts a throwaway one on port
11435 — the daemon gpufree starts in the serve cell owns 11434, and two on one
port is an "address already in use" failure.

The cell ends by asking that daemon to list its models and **fails loudly** if
the import did not take. Skipping this check is how the serve cell ends up
trying to `ollama pull` a model that does not exist upstream:

    Error: pull model manifest: file does not exist

In [ ]:
import json, os, shutil, subprocess, time, urllib.request

work = "/kaggle/working"
model_name = "spark-x2.6-q8-128k"
num_ctx = 131072

# 1. Ollama itself (the serve cell would install it later; we need it now).
if not shutil.which("ollama"):
    if not shutil.which("zstd"):
        subprocess.run("apt-get -qq update && apt-get -qq install -y zstd", shell=True, check=False)
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

# 2. Weights.
from huggingface_hub import hf_hub_download
gguf = hf_hub_download("XHToken/Spark-X2.6-4B-GGUF", "Spark-X2.6-4B-Q8_0.gguf", local_dir=os.path.join(work, "gguf"))
print("gguf:", gguf, round(os.path.getsize(gguf) / 2**30, 2), "GiB")

# 3. Modelfile: the context lives here, not in a cell we have to remember to re-run.
modelfile = os.path.join(work, "Modelfile.spark-x2.6")
with open(modelfile, "w") as fh:
    print("FROM", gguf, file=fh)
    print("PARAMETER num_ctx", num_ctx, file=fh)
    print("PARAMETER num_batch 1024", file=fh)

# 4. Throwaway daemon on our own port, then create + verify + inspect + stop.
env = dict(os.environ, OLLAMA_HOST="127.0.0.1:11435")
logpath = os.path.join(work, "ollama-import.log")
log = open(logpath, "w")
daemon = subprocess.Popen(["ollama", "serve"], env=env, stdout=log, stderr=subprocess.STDOUT)
try:
    for _ in range(90):
        try:
            urllib.request.urlopen("http://127.0.0.1:11435/api/tags", timeout=2)
            break
        except OSError:
            time.sleep(1)
    else:
        raise RuntimeError("ollama did not start on 11435 — see " + logpath)

    created = subprocess.run(["ollama", "create", model_name, "-f", modelfile], env=env, check=False)
    subprocess.run(["ollama", "show", model_name], env=env, check=False)

    # 5. Verify it landed, via the API (the CLI truncates long names in `list`).
    with urllib.request.urlopen("http://127.0.0.1:11435/api/tags", timeout=15) as resp:
        names = [m["name"] for m in json.loads(resp.read()).get("models", [])]
    print("local models:", names)
    if model_name + ":latest" not in names and model_name not in names:
        log.flush()
        print(open(logpath).read()[-4000:])
        raise RuntimeError(
            "ollama create did not register " + model_name + " (exit "
            + str(created.returncode) + "). The log above says why — an unknown "
            "architecture in this Ollama build cannot be fixed from the notebook."
        )
finally:
    daemon.terminate()
    try:
        daemon.wait(timeout=30)
    except subprocess.TimeoutExpired:
        daemon.kill()
    log.close()

# `ollama show` above prints a TEMPLATE section when the GGUF carries a chat
# template. If it came back empty, append chat_template.jinja from the HF repo
# to the Modelfile and re-run `ollama create` — otherwise chat formatting is
# generic and this model's prompt template is wasted.

## Serve it

The env below is what makes 128k usable:

* `OLLAMA_KEEP_ALIVE=-1` — never unload, so a long prompt is prefilled once and
  its KV/prefix is reused across requests instead of re-allocated each time.
* `OLLAMA_NUM_PARALLEL=1` — anything higher splits the 128k window across slots.
* `OLLAMA_KV_CACHE_TYPE=q8_0` — halves the 128k cache (4.6 → 2.3 GiB) and the
  bandwidth attention reads per token. Drop it for f16 if you want the extra
  precision; it still fits.

Verify the context actually took from the log printed by the next cell:
`llama_context: n_ctx = 131072` and two `llama_kv_cache_iswa` lines (one sized
to 131072, one to 512). If `n_ctx` is smaller, the fit was reduced — lower
`num_ctx` in the Modelfile.

The next cell should also print `[gpufree] <model> is already local; skipping
the pull.` If it prints `Pulling ...` instead, the import cell did not take and
the pull will fail.

In [ ]:
import os

os.environ.update(
    OLLAMA_KEEP_ALIVE="-1",
    OLLAMA_FLASH_ATTENTION="1",
    OLLAMA_KV_CACHE_TYPE="q8_0",
    OLLAMA_NUM_PARALLEL="1",
    OLLAMA_MAX_LOADED_MODELS="1",
)

MODEL = "ollama:spark-x2.6-q8-128k"
TUNNEL = "cloudflare"      # "cloudflare" (no account) or "ngrok" (needs authtoken)
PUBLISH = "gist"           # "gist" publishes the endpoint; "none" just prints it

# This cell blocks while the server is alive. First run also loads the model.
!python -m gpufree.server.serve --model {MODEL} --tunnel {TUNNEL} --publish {PUBLISH}

## Use it from your machine

```bash
pipx install --force git+<the repo you cloned above>   # once
gpufree connect --gist                                 # or: --url <url> --key <key>
gpufree chat "summarize this file" < notes.md
gpufree proxy                                          # http://127.0.0.1:8787/v1
```

Long prompts are the point of this notebook: prefill of a genuinely 128k-token
prompt costs a minute or two on a T4, then generation stays fast — the
sliding-window layers mean per-token attention is cheap even at that length.

**Two warnings.** Kaggle meters GPU-hours whether or not the model is busy, so
stop the session when you are done (closing the tab does not stop it). And this
notebook's output contains your API key — Edit > Clear all outputs before any
Save Version.